# Mechanical-bid pattern ATLAS — token-free (Colab)

Open-ended: map the whole **calendar fingerprint** of the passive/401k/rebalance bid and see what patterns
exist, whatever they are — no single hypothesis. Every panel is descriptive: average forward day return by
a calendar feature, with an ascii bar so the SHAPE jumps out on a phone.

Panels: (1) day-of-month shape, (2) day-of-week, (3) first/last-day isolation, (4) month-end vs QUARTER-end
(quarterly pension/TDF rebalance is bigger), (5) month-of-year seasonality, (6) semi-monthly payday (1st vs
15th), (7) options-expiration week. Run top-to-bottom. yfinance→Stooq fallback for SPX.


## Capture setup — tees every printed result into REPORT_LINES for the export cell at the bottom


In [ ]:
import builtins as _bi
REPORT_LINES = []
_realprint = _bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON — run top-to-bottom; the last cell saves everything to a file')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np

def load_spx():
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    try:
        df = yf.download('^GSPC', start='1990-01-01', progress=False, auto_adjust=True)
        if len(df):
            s = df['Close']; s = s.iloc[:,0] if hasattr(s,'columns') else s
            return pd.Series(np.asarray(s).ravel(), index=df.index, name='spx').dropna()
    except Exception as e: print('yf failed -> Stooq:', e)
    import urllib.request, io
    raw = urllib.request.urlopen('https://stooq.com/q/d/l/?s=^spx&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].rename('spx').dropna()

spx = load_spx()
d = pd.DataFrame({'px': spx}); d['ret'] = d['px'].pct_change()
d = d.dropna()
print('SPX', len(d), 'days', d.index[0].date(), '->', d.index[-1].date())


## Tag calendar features + ascii-bar helper


In [ ]:
d['ym'] = d.index.to_period('M'); g = d.groupby('ym')
d['dom_start'] = g.cumcount() + 1
d['dom_end']   = g['px'].transform('size') - g.cumcount() - 1
d['dow']   = d.index.dayofweek          # 0=Mon .. 4=Fri
d['month'] = d.index.month
d['cday']  = d.index.day                 # calendar day-of-month
d['qend_month'] = d['month'].isin([3,6,9,12])
# options expiration week ~ the week containing the 3rd Friday
d['opex_wk'] = (d.index.dayofweek<=4) & (d['cday']>=15) & (d['cday']<=21)

def bar(v, vmax, w=22):
    n = int(round(w*abs(v)/vmax)) if vmax else 0
    return ('█' if v>=0 else '░')*max(n,0)

def show(grp_col, order=None, label='', mincount=30):
    t = d.groupby(grp_col)['ret'].agg(['mean','count'])
    if order is not None: t = t.reindex(order)
    t = t[t['count']>=mincount]
    vmax = (1e4*t['mean'].abs()).max()
    print(f'\n=== {label} ===   (avg bp/day, n)')
    for k,row in t.iterrows():
        bp = 1e4*row['mean']
        print(f'  {str(k):>10} {bp:+6.2f}  {bar(bp,vmax)}  (n={int(row["count"])})')
    return t


## Panel 1 — day-of-month SHAPE (where in the month does the bid live?)


In [ ]:
print('First 22 trading days of the month (dom_start):')
show('dom_start', order=list(range(1,23)), label='avg return by trading-day-of-month')
print('\nLast 3 trading days (dom_end: 0=last):')
show('dom_end', order=[2,1,0], label='avg return, month-END days')


## Panel 2 — day-of-week


In [ ]:
dow_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri'}
tmp = d.copy(); tmp['dow'] = tmp['dow'].map(dow_names)
g2 = tmp.groupby('dow')['ret'].agg(['mean','count']).reindex(['Mon','Tue','Wed','Thu','Fri'])
vmax = (1e4*g2['mean'].abs()).max()
print('=== avg return by day-of-week ===')
for k,row in g2.iterrows(): print(f'  {k} {1e4*row["mean"]:+6.2f}  {bar(1e4*row["mean"],vmax)}  (n={int(row["count"])})')


## Panel 3 — first / last day isolation (the sharpest inflow days)


In [ ]:
first = d[d['dom_start']==1]['ret']; last = d[d['dom_end']==0]['ret']
p1 = d[d['dom_start']==2]['ret']; p2 = d[d['dom_start']==3]['ret']
rest = d[(d['dom_start']>3)&(d['dom_end']>0)]['ret']
for nm,r in [('FIRST day (+1)',first),('2nd day',p1),('3rd day',p2),('LAST day (-1)',last),('mid-month rest',rest)]:
    print(f'  {nm:>16}: {1e4*r.mean():+6.2f} bp/day  win {100*(r>0).mean():.0f}%  ann {100*r.mean()*252:5.1f}%  (n={len(r)})')


## Panel 4 — month-end vs QUARTER-end (quarterly rebalance is bigger)


In [ ]:
tom = (d['dom_start']<=3)|(d['dom_end']==0)
qend_tom = tom & ( ((d['dom_end']==0)&d['qend_month']) | ((d['dom_start']<=3)&d['month'].isin([1,4,7,10])) )
mend_tom = tom & ~qend_tom
for nm,mask in [('QUARTER-end turn',qend_tom),('normal month-end turn',mend_tom),('all other days',~tom)]:
    r = d.loc[mask,'ret']
    print(f'  {nm:>22}: {1e4*r.mean():+6.2f} bp/day  ann {100*r.mean()*252:5.1f}%  (n={len(r)})')
print('\nBigger quarter-end tilt = the quarterly pension/TDF rebalance showing up over the monthly drip.')


## Panel 5 — month-of-year seasonality (annual contribution / bonus / IRA patterns)


In [ ]:
mo_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
mo = d.groupby('month')['ret'].agg(['mean','count'])
mo.index = mo.index.map(mo_names)
vmax = (1e4*mo['mean'].abs()).max()
print('=== avg return by month (annualization aside, this is per-day bp) ===')
for k,row in mo.iterrows(): print(f'  {k} {1e4*row["mean"]:+6.2f}  {bar(1e4*row["mean"],vmax)}  (n={int(row["count"])})')


## Panel 6 — semi-monthly payday proxy (1st vs 15th) + Panel 7 OpEx week


In [ ]:
near1  = d[d['cday']<=2]['ret']; near15 = d[(d['cday']>=14)&(d['cday']<=16)]['ret']
other  = d[(d['cday']>2)&((d['cday']<14)|(d['cday']>16))]['ret']
print('Semi-monthly payday proxy:')
for nm,r in [('around 1st (<=2)',near1),('around 15th (14-16)',near15),('other days',other)]:
    print(f'  {nm:>20}: {1e4*r.mean():+6.2f} bp/day  (n={len(r)})')
opx = d[d['opex_wk']]['ret']; nopx = d[~d['opex_wk']]['ret']
print('\nOptions-expiration week vs not:')
print(f'  OpEx week   : {1e4*opx.mean():+6.2f} bp/day  (n={len(opx)})')
print(f'  non-OpEx    : {1e4*nopx.mean():+6.2f} bp/day  (n={len(nopx)})')


### How to read / caveats
- **Descriptive map, not causation.** A calendar bump is *consistent* with a mechanical bid; it isn't proof
  401k flow caused it. Read the SHAPE, then we reason about the mechanism per panel.
- **Goodhart / decay:** every one of these is public. Add `d = d[d.index>='2018-01-01']` after the load and
  rerun to see which patterns still have a pulse vs which got arbitraged flat.
- **Sample-size discipline:** `n` is printed everywhere. A big bar on a small `n` (e.g. a single day-of-month
  slot) is noise — weight the ones with hundreds of obs.
- Paste any panel back and we'll write the surviving patterns into the ledger. This is the wide net; we tighten
  on whatever it catches.


## ⬇️ EXPORT — run this LAST: saves every result above to one file + downloads it


In [ ]:
# Writes all captured output to mechanical_bid_report.txt and downloads it in Colab.
# Bring that file back to Claude instead of copy-pasting each panel.
from datetime import datetime
fname = 'mechanical_bid_report.txt'
hdr = ['='*70, 'MECHANICAL-BID PATTERN ATLAS — results export',
       f'SPX {d.index[0].date()} -> {d.index[-1].date()}  |  {len(d)} trading days',
       f'generated {datetime.now().strftime("%Y-%m-%d %H:%M")}', '='*70, '']
with open(fname, 'w') as f:
    f.write('\n'.join(hdr) + '\n' + '\n'.join(REPORT_LINES) + '\n')
_realprint('wrote', fname, '—', len(REPORT_LINES), 'lines captured')
if not REPORT_LINES:
    _realprint('!! REPORT_LINES is empty — run the cells ABOVE first (top-to-bottom), then this cell.')
try:
    from google.colab import files
    files.download(fname)
    _realprint('Download started — save it, then upload/paste it back to Claude.')
except Exception as e:
    _realprint('Not in Colab or download blocked:', e)
    _realprint('Open the file browser (folder icon, left margin) and download', fname, 'by hand.')
